In [ ]:
# conda create -p ./.conda python=3.13.5 ipykernel -y; conda activate ./.conda; python -m ipykernel install --user --name "$((Split-Path -Leaf (Get-Location)))-conda" --display-name "Python ($((Split-Path -Leaf (Get-Location)))-conda)"

# Python conversion from Matlab Shock DOE tool

%reset -f 

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import re

########################################################################################

# Functions
def parse_damper_id(id_str):
    # Find positions of delimiters
    if not isinstance(id_str, str) or not id_str:
        raise ValueError("id_str must be a non-empty string")
    
    idxUnder = [m.start() for m in re.finditer(r'_', id_str)]
    idxDash = [m.start() for m in re.finditer(r'-', id_str)]
    idxDot = [m.start() for m in re.finditer(r'\.', id_str)]
    if len(idxDash) < 3 or len(idxDot) < 2:
        raise ValueError("id_str format is incorrect")

    lsc = int(id_str[idxDot[0]+1:idxDash[0]])
    hsc = int(id_str[idxDash[0]+1:idxDash[1]])
    lsr = int(id_str[idxDot[-1]+1:idxDash[-1]])
    hsr = int(id_str[idxDash[-1]+1:])

    compValve = id_str[0:idxDot[0]-2]
    blowoff = id_str[idxDash[1]+1:idxUnder[0]]
    rebValve = id_str[idxUnder[0]+1:idxDot[-1]-2]

    compSpring = id_str[idxDot[0]-2:idxDot[0]]
    rebSpring = id_str[idxDot[-1]-2:idxDot[-1]]

    return compValve, compSpring, lsc, hsc, blowoff, rebValve, rebSpring, lsr, hsr

########################################################################################

# Main script
# clear variables

# ask user for damper ID from clipboard
# Read clipboard data and convert to DataFrame

# ex = """G4750.29-27-0_G4750.21-33	G4750.28-33-0_G4750.14-35
# G4750.29-27-0_G4750.21-33	G4750.15-32-0_G4750.08-36
# G4750.35-25-0_G4750.25-25	G4750.19-25-0_G4750.18-40
# G4750.35-25-0_G4750.25-25	G4750.11-25-0_G4750.20-34"""
# clipboard_data = ex

clipboard_data = input("Please copy the desired data, then press Enter to continue...")

if clipboard_data is None:
    raise ValueError("No data found in clipboard")
df_clipboard = pd.DataFrame([x.split() for x in clipboard_data.splitlines()]) # split by whitespace

num_ids = np.size(df_clipboard, 1)
num_baseline = num_ids//4
if num_baseline == 0:
    num_baseline = np.size(df_clipboard, 1)
num_rows = np.size(df_clipboard, 0)

if (num_ids%4) != 0:
    if num_rows != 4:
        raise ValueError("Expected 4 columns for damper IDs")

arr = df_clipboard.to_numpy().ravel()          # view when possible, faster and no unnecessary copy
if arr.size%4 != 0 or arr.size not in (4,8,12,16):
    raise ValueError(f"expected 4/8/12/16 elements to reshape to (4,3), got {arr.size}")
arr_reshaped = arr.reshape(4,num_baseline)  # reshape to 4 rows, num_baseline columns
damperInput = arr_reshaped

holdSplit = [None] * 4
splitInput = []
for c in range(num_baseline):
    for n in range(4):
        holdSplit[n] = parse_damper_id(str(damperInput[n, c]))
    splitInput.append(np.array(holdSplit).ravel())
splitInput = np.array(splitInput)


iterateProps = True
iterateClicks = True
forceSymmetry = False

numRand = 2500
clickDelta = 10

valveOpt = ['H47', 'G47', 'G67']
springOpt = [20, 30, 40, 50]
lsRange = [1, 60]
hsRange = [1, 40]

typeKey = ['CompValve', 'CompSpring', 'LSC', 'HSC', 'Blowoff', 'RebValve', 'RebSpring', 'LSR', 'HSR']

randValve = np.random.choice(valveOpt, (numRand, 1))
randSpring = np.random.choice(springOpt, (numRand, 1))


for n in range(0, np.size(splitInput,1)):
    if iterateProps:
        if typeKey[n//np.size(typeKey)] in ['CompValve', 'CompSpring', 'RebValve', 'RebSpring']:
            if typeKey[n//np.size(typeKey)] in ['CompValve', 'RebValve']:
                baseValve = splitInput[0,n][0:3]
                randValves = np.random.choice(valveOpt, (numRand, 1))
                if n == 0:
                    randCollect = randValves
                else:
                    randCollect = np.hstack((randCollect, randValves))
            else:
                baseSpring = splitInput[0,n][3:5]
                randSprings = np.random.choice(springOpt, (numRand, 1))
                randCollect = np.hstack((randCollect, randSprings)) 

    if iterateClicks:
        if typeKey[n//np.size(typeKey)] in ['LSC', 'HSC', 'LSR', 'HSR']:
            baseClicks = splitInput[0,n]
            randClicks = np.random.choice(clickDelta, (numRand, 1))
           
    if typeKey[n//np.size(typeKey)] == 'Blowoff':
        randBlow = np.array([0] * numRand).reshape(1,-1)
        
        





# randGen = # first row of split input, same num columns as splitInput
randGen = splitInput[0,:].copy().reshape(1,-1)  # make it 2D with one row
    

id_lf = str(df_clipboard.iloc[0,0])
id_rf = str(df_clipboard.iloc[0,1])
id_lr = str(df_clipboard.iloc[0,2])
id_rr = str(df_clipboard.iloc[0,3])

lf_clicks = parse_damper_id(id_lf)
rf_clicks = parse_damper_id(id_rf)
lr_clicks = parse_damper_id(id_lr)
rr_clicks = parse_damper_id(id_rr)


